# ChatOCIGenAI Standalone Verification

This notebook loads `mcp-client/.env`, initializes `ChatOCIGenAI`, and runs a standalone prompt to verify the OCI Generative AI connection.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

load_dotenv('.env')


In [ ]:
llm = ChatOCIGenAI(
    model_id=os.getenv('MODEL_ID'),
    service_endpoint=os.getenv('SERVICE_ENDPOINT'),
    compartment_id=os.getenv('COMPARTMENT_ID'),
    model_kwargs={
        'temperature': float(os.getenv('MODEL_TEMPERATURE', '0.2')),
        'max_tokens': int(os.getenv('MODEL_MAX_TOKENS', '128')),
    },
    auth_type=os.getenv('AUTH_TYPE'),
    auth_profile=os.getenv('AUTH_PROFILE'),
    provider=os.getenv('PROVIDER'),
)


In [ ]:
response = llm.invoke('Hello from standalone notebook test. Reply with a short greeting.')
response


In [ ]:
mcp_url = os.getenv('MCP_URL')
mcp_url

In [ ]:
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_url = os.getenv('MCP_URL')
if not mcp_url:
    raise ValueError('MCP_URL is not set in the environment.')

client = MultiServerMCPClient(
    {
        'tools_server': {
            'transport': 'http',
            'url': mcp_url,
            'timeout': 30.0,
        },
    }
)

async def run_sentiment(text: str) -> dict:
    async with client.session('tools_server') as session:
        tool_result = await session.call_tool('sentiment_analysis', {'text': text})
    content_blocks = getattr(tool_result, 'content', []) or []
    raw_text = content_blocks[0].text if content_blocks else ''
    return json.loads(raw_text) if raw_text else {}

sentiment_result = await run_sentiment('Enjoying a lot.')
sentiment_result


## Direct Speech MCP Tool Verification (Split Cells)

Run these cells top-to-bottom. Each cell tests one tool.


### Setup
Shared helpers and state for all speech MCP tool tests.


In [ ]:
# Shared setup for split speech tool tests
import os
import json
import asyncio


def _bool_from_env(value: str | None, default: bool = False) -> bool:
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y'}


def _extract_tool_text(tool_result) -> str:
    blocks = getattr(tool_result, 'content', []) or []
    return blocks[0].text if blocks else ''


def _normalize_state(state: str | None) -> str:
    if not state:
        return ''
    return ''.join(ch for ch in str(state).upper() if ch.isalpha())


def _is_success_state(state: str | None) -> bool:
    n = _normalize_state(state)
    return 'SUCCEED' in n


def _is_terminal_state(state: str | None) -> bool:
    n = _normalize_state(state)
    return ('SUCCEED' in n) or ('FAILED' in n) or ('CANCEL' in n)


def _is_cancellable_state(state: str | None) -> bool:
    n = _normalize_state(state)
    if not n:
        return False
    if _is_terminal_state(n):
        return False
    return ('INPROGRESS' in n) or ('ACCEPTED' in n) or ('QUEUED' in n)


async def _call_tool_json(tool_name: str, args: dict | None = None) -> dict:
    args = args or {}
    call_timeout_seconds = int(os.getenv('SPEECH_STATUS_CALL_TIMEOUT_SECONDS', '30'))
    async with client.session('tools_server') as session:
        result = await asyncio.wait_for(session.call_tool(tool_name, args), timeout=call_timeout_seconds)
    raw_text = _extract_tool_text(result)
    if not raw_text:
        return {'raw_result': str(result)}
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        return {'raw_text': raw_text}


speech_test_state = {
    'compartment_id': os.getenv('SPEECH_COMPARTMENT_OCID') or os.getenv('COMPARTMENT_ID'),
    'namespace': os.getenv('OCI_NAMESPACE'),
    'bucket_name': os.getenv('SPEECH_BUCKET'),
    'output_prefix': os.getenv('SPEECH_OUTPUT_PREFIX'),
    'file_names': [f.strip() for f in os.getenv('SPEECH_FILE_NAMES', '').split(',') if f.strip()],
    'verify_existing_only': _bool_from_env(os.getenv('SPEECH_VERIFY_EXISTING_ONLY'), False),
    'created_job_id': '',
    'target_job_id': '',
    'resolved_job_id': '',
    'last_lifecycle_state': '',
    'results': {},
    'jobs_normalized': [],
    'jobs_by_state': {},
    'lookup_object_name': os.getenv('SPEECH_TEST_OBJECT_NAME', '').strip(),
    'found_job_id_from_object': '',
}

missing = [
    name for name, value in {
        'compartment_id (SPEECH_COMPARTMENT_OCID or COMPARTMENT_ID)': speech_test_state['compartment_id'],
        'OCI_NAMESPACE': speech_test_state['namespace'],
        'SPEECH_BUCKET': speech_test_state['bucket_name'],
        'SPEECH_OUTPUT_PREFIX': speech_test_state['output_prefix'],
    }.items() if not value
]
if missing:
    raise ValueError(f'Missing required env vars: {missing}')

speech_test_state






async def _call_tool_raw_text(tool_name: str, args: dict | None = None) -> str:
    args = args or {}
    call_timeout_seconds = int(os.getenv('SPEECH_STATUS_CALL_TIMEOUT_SECONDS', '30'))
    async with client.session('tools_server') as session:
        result = await asyncio.wait_for(session.call_tool(tool_name, args), timeout=call_timeout_seconds)
    return _extract_tool_text(result)



### Tool: list_speech_transcription_jobs
Lists jobs for the given `compartment_id`, optionally narrowed by `bucket_name` and OCI lifecycle filter.


In [ ]:
# Tool test 1: list_speech_transcription_jobs
list_jobs_payload = {
    'compartment_id': speech_test_state['compartment_id'],
    'bucket_name': speech_test_state['bucket_name'],
    'limit': int(os.getenv('SPEECH_LIST_LIMIT', '50')),
    'response_mode': "full",
}
lifecycle_filter = os.getenv('SPEECH_LIST_LIFECYCLE_STATE')
if lifecycle_filter:
    list_jobs_payload['lifecycle_state'] = lifecycle_filter

print('list_jobs_payload:')
print(json.dumps(list_jobs_payload, indent=2))

list_jobs_result = await _call_tool_json(
    'list_speech_transcription_jobs',
    {'payload': json.dumps(list_jobs_payload)}
)

list_jobs_result







In [ ]:

speech_test_state

### Tool: create_speech_transcription_job
Prompts for a local file path, validates OCI-supported format, uploads to bucket, then creates job.


In [ ]:
# Tool test 2: create_speech_transcription_job (upload local file first)
import time
from pathlib import Path
import oci

LOCAL_FILE = Path(os.getenv("LOCAL_SPEECH_FILE", "/Users/<alias>/Downloads/sample-9.mp3"))


if speech_test_state['verify_existing_only']:
    create_result = {'skipped': True, 'reason': 'SPEECH_VERIFY_EXISTING_ONLY=true'}
else:
    supported_exts = {
        '.aac', '.ac3', '.amr', '.au', '.flac', '.m4a', '.mkv',
        '.mp3', '.mp4', '.oga', '.ogg', '.opus', '.wav', '.webm'
    }


    if not LOCAL_FILE:
        raise ValueError('File path is required.')
        LOCAL_FILE = input('Enter local audio/video file path for transcription upload: ').strip()

    local_file = Path(LOCAL_FILE).expanduser()
    if not local_file.exists() or not local_file.is_file():
        raise FileNotFoundError(f'Local file not found: {local_file}')

    ext = local_file.suffix.lower()
    if ext not in supported_exts:
        raise ValueError(
            f'Unsupported file extension: {ext}. Supported: {sorted(supported_exts)}'
        )

    profile = os.getenv('OCI_PROFILE') or os.getenv('AUTH_PROFILE') or 'DEFAULT'
    config_file = os.path.expanduser(os.getenv('OCI_CONFIG_FILE', '~/.oci/config'))
    config = oci.config.from_file(file_location=config_file, profile_name=profile)
    os_client = oci.object_storage.ObjectStorageClient(config)

    object_name = f"uploads/{local_file.name}"
    with local_file.open('rb') as f:
        os_client.put_object(
            namespace_name=speech_test_state['namespace'],
            bucket_name=speech_test_state['bucket_name'],
            object_name=object_name,
            put_object_body=f,
        )

    create_payload = {
        'compartment_id': speech_test_state['compartment_id'],
        'namespace': speech_test_state['namespace'],
        'bucket_name': speech_test_state['bucket_name'],
        'file_names': [object_name],
        'job_name': os.getenv('SPEECH_JOB_NAME'),
        'model_type': os.getenv('SPEECH_MODEL_TYPE', 'WHISPER_LARGE_V3T'),
        'language_code': os.getenv('SPEECH_LANGUAGE_CODE', 'auto'),
        'whisper_prompt': os.getenv('SPEECH_WHISPER_PROMPT'),
        'diarization_enabled': _bool_from_env(os.getenv('SPEECH_DIARIZATION_ENABLED'), True),
        'output_prefix': speech_test_state['output_prefix'],
    }
    create_payload = {k: v for k, v in create_payload.items() if v not in (None, [], '')}

    print('Uploaded object:', object_name)
    print('Create payload:')
    print(json.dumps(create_payload, indent=2))

    create_result = await _call_tool_json(
        'create_speech_transcription_job',
        {'payload': json.dumps(create_payload)}
    )

    speech_test_state['uploaded_object_name'] = object_name
    speech_test_state['uploaded_local_file'] = str(local_file)

speech_test_state['results']['create_speech_transcription_job'] = create_result
created_job_id = (
    create_result.get('job', {}).get('job_id')
    if isinstance(create_result, dict) else ''
) or ''
if created_job_id:
    speech_test_state['created_job_id'] = created_job_id
    speech_test_state['target_job_id'] = created_job_id

create_result


### Tool: get_speech_transcription_job
Fetches lifecycle state/details for the selected job.


In [ ]:
# Tool test 3: get_speech_transcription_job
get_job_result  = {}
if not speech_test_state['target_job_id']:
    get_job_result = {'skipped': True, 'reason': 'No target job id from list/create step'}
else:
    get_job_result = await _call_tool_json(
        'get_speech_transcription_job',
        {'job_id': speech_test_state['target_job_id']}
    )
    print("getting trnscription job status")

speech_test_state['results']['get_speech_transcription_job'] = get_job_result
if isinstance(get_job_result, dict):
    speech_test_state['last_lifecycle_state'] = (
        get_job_result.get('lifecycle_state')
        or get_job_result.get('job', {}).get('lifecycle_state')
        or ''
    )

get_job_result


### Polling Helper
Polls `get_speech_transcription_job` until terminal state or timeout window.


In [ ]:
# Tool test 3b: poll until terminal state and lock resolved job id for downstream tests
import time

max_polls = int(os.getenv('SPEECH_STATUS_MAX_POLLS', '10'))
poll_interval_seconds = int(os.getenv('SPEECH_STATUS_POLL_INTERVAL_SECONDS', '10'))

if not speech_test_state['target_job_id']:
    poll_result = {'skipped': True, 'reason': 'No target job id from list/create step'}
else:
    start_ts = time.time()
    poll_result = {
        'job_id': speech_test_state['target_job_id'],
        'attempts': [],
        'final_state': '',
        'max_polls': max_polls,
        'poll_interval_seconds': poll_interval_seconds,
        'elapsed_seconds': 0,
    }
    for i in range(1, max_polls + 1):
        status_result = await _call_tool_json(
            'get_speech_transcription_job',
            {'job_id': speech_test_state['target_job_id']}
        )
        lifecycle = ''
        if isinstance(status_result, dict):
            lifecycle = (
                status_result.get('lifecycle_state')
                or status_result.get('job', {}).get('lifecycle_state')
                or ''
            )
        poll_result['attempts'].append({'poll': i, 'lifecycle_state': lifecycle})
        poll_result['final_state'] = lifecycle
        speech_test_state['last_lifecycle_state'] = lifecycle

        if _is_success_state(lifecycle):
            speech_test_state['resolved_job_id'] = speech_test_state['target_job_id']
            break

        if _is_terminal_state(lifecycle):
            break

        if i < max_polls:
            await asyncio.sleep(poll_interval_seconds)

    poll_result['elapsed_seconds'] = round(time.time() - start_ts, 1)

speech_test_state['results']['poll_speech_transcription_job'] = poll_result
poll_result


### Tool: get_speech_transcription_text
Returns full transcription text plus `result_object_name` and compact speaker details for agent use.


In [ ]:
# Tool test 4: get_speech_transcription_text (only when SUCCEEDED)
effective_job_id = speech_test_state.get('resolved_job_id') or speech_test_state.get('target_job_id')
if not effective_job_id:
    get_text_result = {'skipped': True, 'reason': 'No target job id'}
elif not _is_success_state(speech_test_state.get('last_lifecycle_state', '')):
    get_text_result = {
        'skipped': True,
        'reason': f"Job is not SUCCEEDED (state={speech_test_state.get('last_lifecycle_state', '')})"
    }
else:
    print('Using job id for get_speech_transcription_text:', effective_job_id)
    raw_text = await _call_tool_raw_text('get_speech_transcription_text', {'job_id': effective_job_id})
    get_text_result = raw_text

speech_test_state['results']['get_speech_transcription_text'] = get_text_result
get_text_result


### Tool: read_transcription_result
Reads/downloads detailed transcription output from Object Storage.


In [ ]:
# Tool test 5: read_transcription_result (download/read response, only when SUCCEEDED)
effective_job_id = speech_test_state.get('resolved_job_id') or speech_test_state.get('target_job_id')
if not effective_job_id:
    read_result = {'skipped': True, 'reason': 'No target job id'}
elif not _is_success_state(speech_test_state.get('last_lifecycle_state', '')):
    read_result = {
        'skipped': True,
        'reason': f"Job is not SUCCEEDED (state={speech_test_state.get('last_lifecycle_state', '')})"
    }
else:
    read_result = await _call_tool_json(
        'read_transcription_result',
        {'job_id': effective_job_id}
    )

speech_test_state['results']['read_transcription_result'] = read_result
read_result




### Tool: cancel_speech_transcription_job
Attempts cancellation using selected job id; server decides if state is cancellable per OCI rules.


In [ ]:
# Tool test 6: cancel_speech_transcription_job
effective_job_id = speech_test_state.get('resolved_job_id') or speech_test_state.get('target_job_id')
if not effective_job_id:
    cancel_result = {'skipped': True, 'reason': 'No target job id'}
else:
    cancel_result = await _call_tool_json(
        'cancel_speech_transcription_job',
        {'job_id': effective_job_id}
    )

speech_test_state['results']['cancel_speech_transcription_job'] = cancel_result
cancel_result


### Tool: list_bucket_audio_files
Lists audio files for the given `namespace`, `bucket_name`, and `prefix`.


In [ ]:
# Tool test 7: list_bucket_audio_files
list_bucket_payload = {
    'compartment_id': speech_test_state['compartment_id'],
    'namespace': speech_test_state['namespace'],
    'bucket_name': speech_test_state['bucket_name'],
    'prefix': os.getenv('SPEECH_LIST_PREFIX', 'uploads/'),
    'limit': int(os.getenv('SPEECH_BUCKET_LIST_LIMIT', '200')),
}
list_bucket_audio_files_result = await _call_tool_json(
    'list_bucket_audio_files',
    {'payload': json.dumps(list_bucket_payload)}
)
speech_test_state['results']['list_bucket_audio_files'] = list_bucket_audio_files_result
list_bucket_audio_files_result


### Tool: find_transcription_job_by_object
Resolves transcription `job_id` from an uploaded object key (for example `uploads/1772770320_sample-2.mp3`) and optionally returns transcript text.


In [ ]:
speech_test_state

In [ ]:
# Tool test 8: find_transcription_job_by_object
# Set SPEECH_TEST_OBJECT_NAME to test a specific file key, e.g. uploads/1772770320_sample-2.mp3

lookup_name = (speech_test_state.get('lookup_object_name') or '').strip()
if not lookup_name:
    latest_files = (speech_test_state.get('results', {}).get('list_bucket_audio_files', {}) or {}).get('files', [])
    if latest_files:
        lookup_name = str(latest_files[0].get('name') or '').strip()

if not lookup_name:
    find_by_object_result = {'skipped': True, 'reason': 'No object name. Set SPEECH_TEST_OBJECT_NAME or run list_bucket_audio_files first.'}
else:
    find_payload = {
        'object_name': lookup_name,
        'compartment_id': speech_test_state['compartment_id'],
        'namespace': speech_test_state['namespace'],
        'bucket_name': speech_test_state['bucket_name'],
        'output_prefix': speech_test_state['output_prefix'],
        'lifecycle_state': os.getenv('SPEECH_FIND_LIFECYCLE_STATE', 'ALL'),
        'include_transcription_text': _bool_from_env(os.getenv('SPEECH_FIND_INCLUDE_TEXT', 'true'), True),
        'max_job_scan': int(os.getenv('SPEECH_FIND_MAX_JOB_SCAN', '1000')),
        'allow_partial_name_match': _bool_from_env(os.getenv('SPEECH_FIND_ALLOW_PARTIAL_MATCH', 'true'), True),
    }
    find_by_object_result = await _call_tool_json(
        'find_transcription_job_by_object',
        {'payload': json.dumps(find_payload)}
    )

    best = find_by_object_result.get('best_match') if isinstance(find_by_object_result, dict) else None
    if isinstance(best, dict):
        found_job_id = str(best.get('job_id') or '').strip()
        if found_job_id:
            speech_test_state['found_job_id_from_object'] = found_job_id
            speech_test_state['target_job_id'] = found_job_id
            speech_test_state['resolved_job_id'] = found_job_id
            speech_test_state['last_lifecycle_state'] = best.get('lifecycle_state') or speech_test_state.get('last_lifecycle_state', '')

speech_test_state['lookup_object_name'] = lookup_name
speech_test_state['results']['find_transcription_job_by_object'] = find_by_object_result
find_by_object_result


In [ ]:
find_payload

### Summary
Shows consolidated results and selected job IDs.


In [ ]:
# Final summary of all split-cell tool checks
speech_test_state['effective_job_id'] = speech_test_state.get('resolved_job_id') or speech_test_state.get('target_job_id')
speech_test_state
